In [0]:
# --------------------------------------------------------------------
# Create widgets for user input (unchanged)
# --------------------------------------------------------------------
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp (e.g., 2023-01-01 00:00:00)")
dbutils.widgets.text("table_name", "", "Target Table Name")
dbutils.widgets.text("load_type", "", "Load Type (FULL, APPEND, MERGE)")
dbutils.widgets.text("merge_keys", "", "Merge Keys (comma‑separated)")

# Retrieve widget values
landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name        = dbutils.widgets.get("table_name")
load_type         = dbutils.widgets.get("load_type").upper()   # normalise
merge_keys        = dbutils.widgets.get("merge_keys")

print(f"[INFO] Widgets loaded → table_name='{table_name}', load_type='{load_type}', "
      f"landing_timestamp='{landing_timestamp}', merge_keys='{merge_keys}'")

In [0]:
# --------------------------------------------------------------------
# Cell 2 – read source (bronze) slice
# --------------------------------------------------------------------
from pyspark.sql import functions as F

# Build the filter expression for readability
filter_expr = (
    F.col("landing_timestamp") == F.to_timestamp(
        F.lit(landing_timestamp),
        "yyyy-MM-dd_HH:mm:ss"
    )
)

print("[INFO] Reading bronze table with filter expression:")
print(f"       ws_adb_caresync.bronze.{table_name} WHERE landing_timestamp = "
      f"to_timestamp('{landing_timestamp}', 'yyyy-MM-dd_HH:mm:ss')")

bronze_df = (
    spark.read.table(f"ws_adb_caresync.bronze.{table_name}")
    .filter(filter_expr)
)

print("[INFO] Bronze slice row count:", bronze_df.count())
display(bronze_df)   # optional: inspect source rows

In [0]:
# --------------------------------------------------------------------
# Cell 3 – add audit columns
# --------------------------------------------------------------------
from pyspark.sql import functions as F

print("[INFO] Adding audit columns (insert_timestamp, update_timestamp)")

updated_df = (
    bronze_df
    .withColumn("insert_timestamp", F.current_timestamp())
    .withColumn("update_timestamp", F.current_timestamp())
)

print("[INFO] Updated dataframe schema after audit columns:")
updated_df.printSchema()
display(updated_df)   # optional: preview before write

In [0]:
# --------------------------------------------------------------------
# Cell 4 – write based on load_type
# --------------------------------------------------------------------
from delta.tables import DeltaTable

target_path = f"ws_adb_caresync.silver.{table_name}"
print(f"[INFO] Target Delta table: {target_path}")

if load_type == "FULL":
    print("[INFO] Load type = FULL → overwrite entire target")
    (
        updated_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_path)
    )
    print("[INFO] FULL write completed")

elif load_type == "APPEND":
    print("[INFO] Load type = APPEND → append to existing target")
    (
        updated_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(target_path)
    )
    print("[INFO] APPEND write completed")

elif load_type == "MERGE":
    print("[INFO] Load type = MERGE → upsert logic start")
    # ----------------------------------------------------------------
    # Ensure the target table exists; create it if not
    # ----------------------------------------------------------------
    if not spark.catalog.tableExists(target_path):
        print("[WARN] Target table does not exist – creating it first (overwrite mode)")
        (
            updated_df
            .write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(target_path)
        )
    else:
        print("[INFO] Target table exists – proceeding with merge")

    # Load the DeltaTable object
    delta_target = DeltaTable.forName(spark, target_path)

    # ----------------------------------------------------------------
    # Build merge condition
    # ----------------------------------------------------------------
    keys = [k.strip() for k in merge_keys.split(",") if k.strip()]
    if not keys:
        raise ValueError("MERGE requires at least one column in merge_keys")
    print(f"[INFO] Merge keys resolved: {keys}")

    # Validate presence of merge keys in source
    missing_keys = [k for k in keys if k not in updated_df.columns]
    if missing_keys:
        raise ValueError(
            f"MERGE keys missing from source data: {', '.join(missing_keys)}"
        )

    # Construct the equality expression used by Delta merge
    merge_cond = " AND ".join([f"src.{k} = tgt.{k}" for k in keys])
    print("[INFO] Generated merge condition:")
    print(f"       {merge_cond}")

    # ----------------------------------------------------------------
    # Execute the merge
    # ----------------------------------------------------------------
    (
        delta_target.alias("tgt")
        .merge(
            source=updated_df.alias("src"),
            condition=merge_cond
        )
        .whenMatchedUpdate(
            set={c: f"src.{c}" for c in updated_df.columns if c != "insert_timestamp"}
        )
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("[INFO] MERGE operation completed")

else:
    raise ValueError(f"Unsupported load_type: {load_type}")

In [0]:
# --------------------------------------------------------------------
# Cell 5 – return record count (same for all modes)
# --------------------------------------------------------------------
records_count = updated_df.count()
print(f"[INFO] Final record count (source after audit cols): {records_count}")

# Exit the notebook with the count – can be captured by a parent workflow
dbutils.notebook.exit(str(records_count))